# AdwenTTS on a free GPU

Runs the AdwenTTS voice-cloning worker on Google Colab's free T4.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. The last cell prints a public link to a web page where you upload a reference clip, type text and download the audio.

Free Colab sessions time out after a few idle hours and when the browser closes. Nothing is lost except the running server; re-run the cells to come back.

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime type to T4 GPU before continuing" 

## Install

Installs the worker with the Chatterbox engine straight from GitHub, pinning PyTorch, torchvision and torchaudio to matching versions (Chatterbox requires torch 2.6.0 and Colab ships a newer torchvision that would otherwise break the import). Takes 3 to 5 minutes the first time. The last line must print `Installed adwentts_worker 0.1.0` and `cuda: True`. If Colab shows a restart banner instead, restart the runtime, skip this cell and continue below.

In [ ]:
# Chatterbox pins torch 2.6.0; torchvision/torchaudio must match or transformers fails to import.
%pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    "adwentts-worker[chatterbox,ui] @ git+https://github.com/sdanquah101/adwenTTS.git@main#subdirectory=worker"

import importlib
importlib.invalidate_caches()
import torch, torchvision, adwentts_worker
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda:", torch.cuda.is_available())
print("Installed adwentts_worker", adwentts_worker.__version__)

## Download the model

First run fetches about 2 GB of weights from the Hugging Face Hub and caches them for this session.

In [ ]:
import os
os.environ["ADWENTTS_ENGINE"] = "chatterbox"   # or "chatterbox-turbo" for faster, slightly lower quality
os.environ["ADWENTTS_VOICES_DIR"] = "/content/voices"

from adwentts_worker.engines import create_engine
from adwentts_worker.service import SynthesisService
from adwentts_worker.voices import VoiceStore

engine = create_engine(os.environ["ADWENTTS_ENGINE"])
engine.load()
service = SynthesisService(engine, VoiceStore(os.environ["ADWENTTS_VOICES_DIR"]))
print("Ready:", engine.name, "on", engine.device)

## Option A: web page with a public link

Upload or record a 10 to 20 second clip of the voice, type the text, click Generate, download the file. The cell keeps running while the page is live; stop it to close the link.

In [ ]:
from adwentts_worker.ui import build_ui
build_ui(service).launch(share=True)

## Option B: generate files directly in the notebook

Upload a reference clip, synthesize, and download the result.

In [ ]:
from google.colab import files
from pathlib import Path
from adwentts_worker.engines import SynthesisOptions

uploaded = files.upload()                      # pick your reference clip
ref = Path(next(iter(uploaded)))

text = """Akwaaba. Welcome to Accra, the capital of Ghana.
We are glad you could join us today."""

result = service.synthesize(text, reference_path=ref, fmt="mp3", options=SynthesisOptions(exaggeration=0.5, cfg_weight=0.5))
out = Path("speech.mp3"); out.write_bytes(result.audio)
print(f"{result.duration_seconds:.1f}s of audio in {result.elapsed_seconds:.1f}s")
files.download(str(out))

## Option C: public API for AI models

Starts the REST + MCP API inside this runtime (reusing the loaded model) and opens a free Cloudflare tunnel so it is reachable from anywhere. The cell prints the base URL and an API key to give to your model. Both change every time this session restarts. See `docs/AGENT_INTEGRATION.md` in the repo for how to connect Claude, MCP clients or OpenAI-style tools.

The Colab "port proxy" links (`*.prod.colab.dev`) only work in your own browser, which is why a tunnel is needed.

In [ ]:
import re, secrets, subprocess, threading, time, urllib.request
from pathlib import Path
import uvicorn
from adwentts_worker.outputs import OutputStore
from adwentts_worker.server import create_app
from adwentts_worker.settings import Settings

PORT = 8000
API_KEY = secrets.token_urlsafe(24)

# 1. Free public tunnel (Cloudflare quick tunnel, no account needed)
cloudflared = Path("/usr/local/bin/cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    cloudflared.chmod(0o755)
tunnel_log = Path("/content/cloudflared.log")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log.open("w"), stderr=subprocess.STDOUT,
)
public_url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text())
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)
assert public_url, "tunnel did not start; see /content/cloudflared.log"

# 2. API server in this runtime, sharing the already-loaded engine
settings = Settings(
    engine=engine.name, api_key=API_KEY, public_url=public_url, port=PORT,
    voices_dir=Path("/content/voices"), output_dir=Path("/content/outputs"),
)
app = create_app(engine=engine, voices=service.voices, outputs=OutputStore(settings.output_dir), settings=settings)
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health")
        break
    except Exception:
        time.sleep(1)

print(f"""Base URL : {public_url}
API key  : {API_KEY}
Docs     : {public_url}/docs
OpenAPI  : {public_url}/openapi.json
MCP      : {public_url}/mcp

Try it from any terminal:
curl -s -X POST {public_url}/v1/tts -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
  -d '{{"text": "Akwaaba, welcome to Accra.", "format": "mp3", "response": "json"}}'
""")

### Add voices

Models cannot upload reference clips, so save the voices here. Upload one clip per speaker (10 to 20 seconds, quiet room) and name it. The model then refers to the voice by name.

In [ ]:
from google.colab import files
from pathlib import Path

name = input("Voice name (e.g. Kofi): ").strip()
uploaded = files.upload()
clip = Path(next(iter(uploaded)))
voice = service.voices.add(name, clip)
print(f"Saved voice {voice.name!r} id={voice.id} ({voice.duration_seconds}s)")
print("All voices:", [v.name for v in service.voices.list()])

### Keep the session alive

Colab stops idle runtimes. Run this cell and leave it running while your model is using the API; stop it when you are done.

In [ ]:
import time, urllib.request
while True:
    status = urllib.request.urlopen(f"http://localhost:{PORT}/health").read().decode()
    print(time.strftime("%H:%M:%S"), "alive", status[:60])
    time.sleep(300)